# 04 — LLM-Pipeline 1: Strukturiertes JSON → natürlichsprachliche Erklärung

Das LLM erhält globale Feature-Importance und lokale SHAP-/EBM-Beiträge als **strukturiertes JSON**
und formuliert daraus eine deutsche Erklärung für nicht-technische Nutzer.

- **Vorteil:** Präzise, maschinenlesbare Eingabe.
- **Nachteil:** Kein visueller Kontext (keine Plots).

Pro Kombination (XGBoost/EBM × 5 Instanzen) wird eine Erklärung generiert
und in `results/pipeline04/` gespeichert.

In [ ]:
from __future__ import annotations

import sys, json, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import (
    INSTANCE_IDS, N_GENERATIONS_SCALE, scale_instance_ids,
    EXPLANATIONS_DIR, RESULTS_DIR, PROMPTS_DIR,
)
from utils.llm import ask_text, DEFAULT_MODEL, MAX_TOKENS_GENERATION, strip_scratchpad

LOSS_KEY   = "poisson_log"
MODEL      = DEFAULT_MODEL
MAX_TOKENS = MAX_TOKENS_GENERATION  # B6: angehoben (Scratchpad-Overhead)

# ── Phase 3b — Skalierungsschalter ────────────────────────────────────────────
#   SCALE_RUN = False → n=20-Validität: die 10 Instanzen, 1 Generation, real-time,
#                       Ausgabe direkt in results/pipeline04/ (…_inst{iid}.json).
#   SCALE_RUN = True  → n≈200 × N Generationen (seeded), als EIN Anthropic-Batch
#                       (−50 %), Ausgabe in results/pipeline04/scale/ — physisch
#                       getrennt von der Validität (…_inst{iid}_gen{g}.json).
SCALE_RUN = True
if SCALE_RUN:
    GEN_INSTANCE_IDS = scale_instance_ids()   # 200, deterministisch (seed=42)
    N_GEN            = N_GENERATIONS_SCALE     # 3 Generationen/Instanz
    OUT_DIR          = RESULTS_DIR / "pipeline04" / "scale"
else:
    GEN_INSTANCE_IDS = INSTANCE_IDS
    N_GEN            = 1
    OUT_DIR          = RESULTS_DIR / "pipeline04"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"LLM-Modell:    {MODEL}")
print(f"Loss-Option:   {LOSS_KEY}")
print(f"SCALE_RUN:     {SCALE_RUN}  (n={len(GEN_INSTANCE_IDS)} Instanzen × {N_GEN} Generationen)")
print(f"Ausgabe:       {OUT_DIR}")

## 1 · System-Prompt

Der System-Prompt wird gecacht (Anthropic Prompt Caching). Da das Minimum
für einen Cache-Block 1 024 Tokens beträgt, enthält der System-Prompt neben
den Instruktionen auch das Feature-Schema mit vollständigen Beschreibungen —
so überschreitet er die Schwelle zuverlässig und wird bei allen Folge-Aufrufen
aus dem Cache gelesen.


In [2]:
SYSTEM_PROMPT = (PROMPTS_DIR / "pipeline_04_json.md").read_text()

print(f"System-Prompt: {len(SYSTEM_PROMPT)} Zeichen, ~{len(SYSTEM_PROMPT)//4} Tokens (geschätzt)")

System-Prompt: 3028 Zeichen, ~757 Tokens (geschätzt)


## 2 · Hilfsfunktionen

`build_context_string` übersetzt normalisierte Rohwerte in lesbare Angaben
(z.B. `temp=0.68` → `~27.9 °C`). Diese Kontextzeile wird dem JSON-Payload
als `human_readable_context` beigefügt, damit das Modell nicht selbst
denormalisieren muss.


In [3]:
from utils.explanations import build_context_string

def load_explanations(model_name: str, instance_id: int) -> tuple[dict, dict]:
    g = json.loads(
        (EXPLANATIONS_DIR / f"global_{model_name}_{LOSS_KEY}.json").read_text()
    )
    l = json.loads(
        (EXPLANATIONS_DIR / f"local_{model_name}_{LOSS_KEY}_inst{instance_id}.json").read_text()
    )
    return g, l


def build_user_prompt(global_exp: dict, local_exp: dict, top_k: int = 5) -> str:
    fv = local_exp["feature_values"]
    payload = {
        "model": local_exp["model"],
        "metrics": {
            "rmse":             global_exp["metrics"]["rmse"],
            "r2":               global_exp["metrics"]["r2"],
            "poisson_deviance": global_exp["metrics"]["poisson_deviance"],
        },
        "global_top_features": [
            {"rank": e["rank"], "feature": e["feature"]}
            for e in global_exp["global_importance"][:top_k]
        ],
        "instance_id":           local_exp["instance_id"],
        "feature_values":        fv,
        "human_readable_context": build_context_string(fv),
        "y_true":                local_exp["y_true"],
        "prediction":            local_exp["prediction"],
        "top_contributions":     local_exp["contributions"][:6],
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


# Beispiel-Prompt anzeigen
g, l = load_explanations("xgb", INSTANCE_IDS[0])
sample_prompt = build_user_prompt(g, l)
print(f"Beispiel User-Prompt ({len(sample_prompt)} Zeichen):")
print(sample_prompt)


Beispiel User-Prompt (1374 Zeichen):
{
  "model": "xgb",
  "metrics": {
    "rmse": 45.439511,
    "r2": 0.935358,
    "poisson_deviance": 9.380072
  },
  "global_top_features": [
    {
      "rank": 1,
      "feature": "hr"
    },
    {
      "rank": 2,
      "feature": "yr"
    },
    {
      "rank": 3,
      "feature": "temp"
    },
    {
      "rank": 4,
      "feature": "weekday"
    },
    {
      "rank": 5,
      "feature": "mnth"
    }
  ],
  "instance_id": 224,
  "feature_values": {
    "weathersit": 1.0,
    "mnth": 2.0,
    "hr": 19.0,
    "weekday": 4.0,
    "yr": 0.0,
    "holiday": 0.0,
    "temp": 0.2,
    "hum": 0.4,
    "windspeed": 0.0
  },
  "human_readable_context": "19:00 Uhr, Donnerstag, Februar, 2011, klar/wenige Wolken, ~8.2 °C, 40 % Luftfeuchtigkeit, Wind 0.0 km/h",
  "y_true": 96.0,
  "prediction": 111.9967,
  "top_contributions": [
    {
      "feature": "temp",
      "value": 0.2,
      "contribution": -0.548818
    },
    {
      "feature": "hr",
      "val

## 3 · LLM-Aufrufe

> **Voraussetzung:** `ANTHROPIC_API_KEY` muss gesetzt sein:
> ```bash
> export ANTHROPIC_API_KEY=sk-ant-...
> ```

In [ ]:
from utils import run_resumable_generation, run_batch_generation
from utils.llm import build_text_params, run_params

# Phase 3a·B / 3b — Ausführungsart folgt dem Skalierungsschalter (Zelle 1).
#   USE_BATCHES = SCALE_RUN: der Vollauf (n≈200 × N) geht als EIN Anthropic-Batch
#   (−50 %); die kleine n=20-Validität bleibt real-time. Beide Pfade schreiben über
#   denselben build_record dieselben Dateien (Golden-Vergleich test_batch_generation).
USE_BATCHES = SCALE_RUN

totals = {"in": 0, "out": 0, "cache": 0}


def build_request(model_name, iid, gen_idx):
    """messages.create-Params für eine Erklärung — gemeinsam für batch & real-time."""
    g_exp, l_exp = load_explanations(model_name, iid)
    user_prompt = build_user_prompt(g_exp, l_exp)
    return build_text_params(
        user_prompt,
        system=SYSTEM_PROMPT,
        model=MODEL,
        max_tokens=MAX_TOKENS,
        cache_system=True,
    )


def build_record(model_name, iid, gen_idx, text, usage, elapsed_s=None):
    """Baut den persistierten Record — identisch für batch- und real-time-Pfad."""
    text    = strip_scratchpad(text)
    in_tok  = usage.get("input_tokens", 0)
    out_tok = usage.get("output_tokens", 0)
    cache_r = usage.get("cache_read_input_tokens", 0)
    totals["in"] += in_tok; totals["out"] += out_tok; totals["cache"] += cache_r

    _, l_exp = load_explanations(model_name, iid)
    return {
        "pipeline":    "04_json",
        "llm_model":   MODEL,
        "loss_key":    LOSS_KEY,
        "xai_model":   model_name,
        "instance_id": iid,
        "explanation": text,
        "elapsed_s":   elapsed_s,
        "usage":       {"input_tokens": in_tok, "output_tokens": out_tok,
                        "cache_read_input_tokens": cache_r},
        "prediction":  l_exp["prediction"],
        "y_true":      l_exp["y_true"],
    }


def generate_json(model_name, iid, gen_idx):
    """Real-time-Generierung einer Einheit (wird vom Resume-Loop gerufen)."""
    t0 = time.time()
    response = run_params(build_request(model_name, iid, gen_idx))
    elapsed  = time.time() - t0

    text   = response["content"][0]["text"]
    usage  = response.get("usage", {})
    record = build_record(model_name, iid, gen_idx, text, usage, elapsed_s=round(elapsed, 2))

    print(f"  {model_name.upper()} inst={iid:4d} g{gen_idx}  "
          f"pred={record['prediction']:6.1f}  y={record['y_true']:5.0f}  "
          f"in={record['usage']['input_tokens']}  out={record['usage']['output_tokens']}  "
          f"cache={record['usage']['cache_read_input_tokens']}  t={elapsed:.1f}s")
    return record


def _on_skip(record, model_name, iid, gen_idx):
    pass  # bei n≈200 nicht je übersprungene Einheit loggen (zu viel Output)


if USE_BATCHES:
    # Fehlende Einheiten (skip-if-exists) als einen Batch einreichen.
    results = run_batch_generation(
        model_names=["xgb", "ebm"],
        instance_ids=GEN_INSTANCE_IDS,
        out_dir=OUT_DIR,
        build_request=build_request,
        build_record=build_record,
        pipeline_tag="gen04",
        n_generations=N_GEN,
        state_path=OUT_DIR / "_batch_state.json",
        on_skip=_on_skip,
    )
else:
    results = run_resumable_generation(
        model_names=["xgb", "ebm"],
        instance_ids=GEN_INSTANCE_IDS,
        out_dir=OUT_DIR,
        generate=generate_json,
        n_generations=N_GEN,
        on_skip=_on_skip,
    )

print(f"\nGesamt:  input={totals['in']}  output={totals['out']}  cache_gelesen={totals['cache']}  "
      f"({len(results)} Einheiten)")

## 4 · Beispiel-Erklärungen

In [5]:
for rec in results[:2]:
    sep = '=' * 70
    print(sep)
    print(f"Modell: {rec['xai_model'].upper()}  |  Instanz: {rec['instance_id']}  "
          f"|  Vorhersage: {rec['prediction']:.1f}  |  Tatsächlich: {rec['y_true']:.0f}")
    print(sep)
    print(rec["explanation"])
    print()

Modell: XGB  |  Instanz: 224  |  Vorhersage: 112.0  |  Tatsächlich: 96
Das Modell sagt für diese Stunde 112 ausgeliehene Fahrräder vorher; tatsächlich waren es 96. Die Abweichung von rund 16 Fahrrädern entspricht etwa 17 Prozent — das ist eine mäßige Vorhersage, die die Größenordnung gut trifft, aber die Nachfrage etwas überschätzt.

Der mit Abstand stärkste Bremsfaktor in dieser Stunde ist die Temperatur: Mit knapp 8 °C ist es spürbar kalt, und das schreckt viele Nutzerinnen und Nutzer vom Radfahren ab. Dieser Kälteeffekt drückt die erwartete Nachfrage deutlich nach unten. Verstärkt wird das durch den Monat Februar, der saisonal zu den schwächsten Monaten gehört — die Kombination aus Kälte und Wintermonat erklärt, warum die Zahlen trotz eines günstig gelegenen Feierabendzeit-Effekts moderat bleiben. Dass wir uns im Jahr 2011 befinden — dem ersten Betriebsjahr des Systems — wirkt sich ebenfalls dämpfend aus, da das Verleihsystem noch weniger bekannt war und die Nutzerbasis kleiner war.

## 5 · Zusammenfassung

In [6]:
import pandas as pd

summary = pd.DataFrame([
    {
        "Modell":     r["xai_model"].upper(),
        "Instanz":    r["instance_id"],
        "y_true":     r["y_true"],
        "Vorhersage": r["prediction"],
        "Wörter":     len(r["explanation"].split()),
        "tok_input":  r["usage"]["input_tokens"],
        "tok_output": r["usage"]["output_tokens"],
        "Zeit (s)":   r["elapsed_s"],
    }
    for r in results
])
display(summary)

,Modell,Instanz,y_true,Vorhersage,Wörter,tok_input,tok_output,Zeit (s)
0,XGB,224,96.0,111.9967,217,615,532,12.87
1,XGB,580,63.0,64.5773,184,616,463,11.20
2,XGB,1041,387.0,385.3899,200,615,491,10.38
3,XGB,1481,277.0,228.5813,198,617,492,11.64
4,XGB,1677,286.0,254.2623,209,615,498,10.76
5,XGB,2058,243.0,194.0592,208,616,498,10.83
6,XGB,2510,372.0,326.0351,209,619,536,12.65
7,XGB,3543,286.0,309.9225,203,618,505,11.11
8,XGB,3847,531.0,585.7666,204,617,504,10.98
9,XGB,4454,354.0,297.3312,209,617,516,11.42
